# C5-neural-networks — Session 1: Perceptrons and Activations

*One class session, roughly 80 minutes. Prerequisites: C2-linear-models (the
linear model $\hat y_i = \sum_k X_{ik} w_k + b$ in component form),
C3-gradient-descent (what training a linear model looked like — for contrast),
and F4-multivar-calculus (the $\tanh$ function and its derivative
$1 - \tanh^2$).*

**This session:** the smallest neural building block.
A **perceptron** is C2's weighted sum with a decision bolted on: compute
$z = \sum_k w_k x_k + b$, then answer *yes* (1) or *no* (0) by thresholding
$z$ at zero.
Today: the perceptron and its firing rule, the family of **activation
functions** (step, $\tanh$, ReLU) that turn sums into decisions and shapes,
what ReLU combinations can build — and the theorem-shaped fact that makes all
of this necessary: a stack of linear maps with no activation in between is
still just one linear map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804

## 1. From Weighted Sum to Decision: the Perceptron

**Motivation.**
Every model since C2 has *output a number*: a predicted price, a loss, a
gradient entry.
But many questions want a *decision* — spam or not, inside or outside,
admit or reject.
The oldest neural answer: keep the weighted sum, then **threshold** it.

**Definition.**
A **perceptron** with weights $w = (w_1, \dots, w_d)$ and bias $b$ maps an
input $x = (x_1, \dots, x_d)$ to

$$z \;=\; \sum_{k=1}^{d} w_k x_k + b
\qquad\text{(the pre-activation)},$$

$$\text{output} \;=\;
\begin{cases} 1 & \text{if } z \ge 0 \\ 0 & \text{if } z < 0 \end{cases}
\qquad\text{(the activation).}$$

Vocabulary, pinned for the whole unit: $z$ is the **pre-activation**, the
0-or-1 result is the unit's **activation** or output, and the unit **fires**
when it outputs 1.
The boundary convention is part of the contract: $z = 0$ **fires** (output
1) — this course and its exam register always threshold with $\ge$.

**Worked example, by hand.**
Take $w = (2, -1)$, $b = -3$, so $z = 2 x_1 - x_2 - 3$:

| $x$ | $z = 2x_1 - x_2 - 3$ | output |
|---|---|---|
| $(2, 0)$ | $4 - 0 - 3 = 1$ | 1 |
| $(1, 1)$ | $2 - 1 - 3 = -2$ | 0 |
| $(0, 3)$ | $0 - 3 - 3 = -6$ | 0 |
| $(3, 3)$ | $6 - 3 - 3 = 0$ | **1** (boundary fires) |

> **Where do the weights come from?**
> In this unit: from *you*.
> Training multi-layer networks needs backpropagation, which is beyond this
> course's scope; the exam tests **inference engineering** — you will design
> weights by hand to make networks compute exactly what you specify.
> (C3 trained *linear* models by gradient descent; nothing in this unit is
> trained.)
> Session 3 covers the one place randomness replaces design: choosing the
> *scale* of random weights.

In [ ]:
# the worked example, in the course's component form (C2 register)
w = np.array([2.0, -1.0])
b = -3.0
pts = np.array([[2.0, 0.0], [1.0, 1.0], [0.0, 3.0], [3.0, 3.0]])

z = (pts * w).sum(axis=1) + b          # z_i = sum_k pts[i, k] * w_k + b
out = (z >= 0).astype(float)
print("z      :", z)
print("outputs:", out)

The prints match the hand table: $z = (1, -2, -6, 0)$ and outputs
$(1, 0, 0, 1)$ — the boundary point $(3, 3)$ fires.

### Checkpoint 1

1. For the same unit ($w = (2,-1)$, $b = -3$): compute $z$ and the output
   at $(1, -2)$ and at $(2, 1)$.
2. Design integer weights and bias so that a perceptron on one input $x_1$
   fires exactly when $x_1 \ge 2$.
3. Multiply $w$ and $b$ of Checkpoint 1's unit by $10$.
   What happens to each $z$, and what happens to each output? One sentence.

## 2. The Threshold Activation, as a Function

**Definition.**
Separate the thresholding from the particular perceptron: the **threshold
(step) activation** is the function

$$\mathrm{step}(z) = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0,\end{cases}$$

applied **elementwise** to arrays.
In code it is one comparison plus a dtype conversion, and its name is pinned
unit-wide (C6-pytorch rebuilds it under the same contract):

In [ ]:
def step_activation(z):
    """Threshold activation, elementwise: 1.0 where z >= 0, else 0.0."""
    return (z >= 0).astype(float)


print(step_activation(np.array([-1.5, -0.0, 0.0, 0.3, 2.0])))

zs = np.linspace(-3, 3, 601)
plt.figure(figsize=(5.5, 2.8))
plt.plot(zs, step_activation(zs))
plt.scatter([0], [1], color="C3", zorder=3, label="step(0) = 1")
plt.xlabel("z")
plt.ylabel("step(z)")
plt.title("The threshold activation")
plt.legend()
plt.show()

All five prints are decided by the sign test — including `-0.0`, which
compares `>= 0` as true, so the output is $(0, 1, 1, 1, 1)$.

**The geometry a thresholded sum creates.**
A perceptron's firing set $\{x : z(x) \ge 0\}$ is one side of the straight
line $z(x) = 0$ (in two dimensions; a plane in three, and so on).
For the Section 1 unit the boundary is $2x_1 - x_2 - 3 = 0$, i.e.
$x_2 = 2x_1 - 3$; the unit fires on the side where $z > 0$ (below the
line — larger $x_1$, smaller $x_2$):

In [ ]:
rng = np.random.default_rng(SEED)
cloud = rng.uniform([-1, -4], [4, 6], (300, 2))
fire = step_activation((cloud * w).sum(axis=1) + b)

plt.figure(figsize=(5.5, 4))
plt.scatter(cloud[fire == 1, 0], cloud[fire == 1, 1], s=10, label="fires (z >= 0)")
plt.scatter(cloud[fire == 0, 0], cloud[fire == 0, 1], s=10, label="silent (z < 0)")
xs = np.linspace(-1, 4, 50)
plt.plot(xs, 2 * xs - 3, color="k", linewidth=1, label="boundary  $x_2 = 2x_1 - 3$")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("One perceptron = one straight boundary")
plt.legend(loc="upper left", fontsize=8)
plt.show()
print("fraction firing:", fire.mean())

**Why the step resists training — and why that is fine here.**
The step is flat everywhere except the jump: its derivative is $0$ at every
$z \ne 0$.
A C3 gradient step $-\eta \, \partial L / \partial w$ therefore receives no
signal through a step activation — one more reason this unit *designs*
weights instead of fitting them.
Session 2 turns the one-line geometry above into a design tool.

### Checkpoint 2

1. Evaluate `step_activation` by hand on $(-0.2,\; 0.0,\; 5.0,\; -7.1)$.
2. For the plotted unit, does the point $(3, 1)$ fire? Decide from the
   inequality, not the picture.
3. In one sentence: why does a gradient-descent step get no information
   through a step activation at $z = 1.7$?

## 3. The Activation Family: Step, tanh, ReLU

**Definition.**
An **activation function** is a fixed scalar nonlinearity applied
elementwise to pre-activations.
The three this course uses:

- **Step** — the threshold above: outputs in $\{0, 1\}$, exact decisions.
- **tanh** — F4's smooth squasher: outputs in $(-1, 1)$, with
  $\tanh(0) = 0$ and derivative $1 - \tanh^2(z)$ (F4's identity).
  Large $|z|$ saturates toward $\pm 1$.
- **ReLU** (rectified linear unit) — new, and the workhorse of modern
  networks:

$$\mathrm{relu}(z) = \max(z, 0) =
\begin{cases} z & z \ge 0 \\ 0 & z < 0.\end{cases}$$

ReLU passes positive values through *unchanged* and silences negative ones.
It is not bounded, and it is piecewise linear with a single kink at $0$.

In [ ]:
def relu(z):
    """ReLU activation, elementwise: max(z, 0)."""
    return np.maximum(z, 0.0)


probe = np.array([-2.0, -0.5, 0.0, 0.5, 3.0])
print("step:", step_activation(probe))
print("relu:", relu(probe))
print("tanh:", np.round(np.tanh(probe), 4))

plt.figure(figsize=(6, 3.2))
plt.plot(zs, step_activation(zs), label="step")
plt.plot(zs, np.tanh(zs), label="tanh")
plt.plot(zs, relu(zs), label="relu")
plt.xlabel("z")
plt.ylabel("activation(z)")
plt.title("The three activations")
plt.legend()
plt.show()

Read the probe prints against the definitions: step gives
$(0, 0, 1, 1, 1)$; relu gives $(0, 0, 0, 0.5, 3)$; tanh gives
$(-0.964, -0.4621, 0, 0.4621, 0.9951)$ — squashed, odd-symmetric, saturating.

The comparison table to memorize:

| | range | value at $0$ | derivative | character |
|---|---|---|---|---|
| step | $\{0, 1\}$ | $1$ (fires) | $0$ for all $z \ne 0$ | exact yes/no |
| tanh | $(-1, 1)$ | $0$ | $1 - \tanh^2(z)$, max $1$ at $0$ | smooth squash |
| relu | $[0, \infty)$ | $0$ | $0$ for $z<0$, $1$ for $z>0$, kink at $0$ | one-sided identity |

### Checkpoint 3

1. Evaluate all three activations at $z = -1$, $0$, and $2$
   ($\tanh$ to three decimals: $\tanh 1 \approx 0.762$,
   $\tanh 2 \approx 0.964$).
2. Which of the three are bounded? Which is differentiable everywhere?
3. Using F4's identity, compute the derivative of $\tanh$ at $z = 0$ and
   explain in one sentence why tanh behaves almost linearly for small $|z|$.

## 4. ReLU Combinations Build Piecewise-Linear Shapes

**Motivation.**
Step units answer yes/no; ReLU units build *shapes*.
Sums of scaled, shifted ReLUs are exactly the piecewise-linear functions —
this is the geometric reason deep ReLU networks are so expressive, and it is
fully visible in one dimension.

**The ramp.**
For $a < b$,

$$\mathrm{ramp}(z) = \mathrm{relu}(z - a) - \mathrm{relu}(z - b)$$

is $0$ for $z \le a$, rises with slope $1$ on $[a, b]$, and stays constant at
$b - a$ for $z \ge b$: the two slopes-of-one cancel beyond $b$.
Worked with $a = 0$, $b = 2$: at $z = -1, 1, 3$ the values are
$0 - 0 = 0$, $\;1 - 0 = 1$, $\;3 - 1 = 2$.

**Slopes add.**
Each term $c \cdot \mathrm{relu}(z - a)$ contributes slope $c$ to every
$z > a$ and nothing before $a$; kinks sit at the shift points.
So $h(z) = 2\,\mathrm{relu}(z) - 3\,\mathrm{relu}(z - 1)$ has slope $0$
before $0$, slope $2$ on $(0, 1)$, and slope $2 - 3 = -1$ after $1$:
up to $h(1) = 2$, then drifting down through $h(3) = 6 - 6 = 0$.

In [ ]:
zg = np.linspace(-2, 4, 601)
ramp = relu(zg - 0.0) - relu(zg - 2.0)
h = 2 * relu(zg) - 3 * relu(zg - 1.0)

for zq in (-1.0, 1.0, 3.0):
    r = relu(zq - 0.0) - relu(zq - 2.0)
    hv = 2 * relu(zq) - 3 * relu(zq - 1.0)
    print(f"z = {zq:4.1f}   ramp = {r:.1f}   h = {hv:.1f}")

plt.figure(figsize=(6, 3.0))
plt.plot(zg, ramp, label="ramp: relu(z) - relu(z-2)")
plt.plot(zg, h, label="h: 2 relu(z) - 3 relu(z-1)")
plt.xlabel("z")
plt.ylabel("value")
plt.title("ReLU sums are piecewise-linear")
plt.legend(fontsize=8)
plt.show()

The prints confirm the hand values — ramp $(0, 1, 2)$ and
$h = (0, 2, 0)$ at $z = (-1, 1, 3)$.
Practice p07 builds a staircase this way, and the challenge p18 runs the
logic in reverse: given a target piecewise-linear function, *design* the
ReLU network that equals it exactly.

### Checkpoint 4

1. For $g(z) = \mathrm{relu}(z + 1) - \mathrm{relu}(z - 1)$: compute
   $g(-2), g(0), g(4)$, and describe $g$ piece by piece.
2. Write a ReLU combination that is $0$ for $z \le 0$, has slope $1$ on
   $[0, 3]$, and is constant for $z \ge 3$. What constant?
3. Where are the kinks of $q(z) = \mathrm{relu}(z) + 4\,\mathrm{relu}(z-2)
   - \mathrm{relu}(z-5)$, and what is the slope on each piece?

## 5. Why Nonlinearity Matters: a Stack of Linear Maps Is Linear

**Motivation.**
If activations complicate everything, why not stack plain affine layers?
Because stacking buys *nothing* without a nonlinearity in between — a fact
worth working in full, in this course's component form.

**Setup.**
Layer 1 sends $x \in \mathbb{R}^{d}$ to
$z^{(1)}_j = \sum_k W^{(1)}_{jk} x_k + b^{(1)}_j$; layer 2, *reading
$z^{(1)}$ directly with no activation*, outputs
$z^{(2)}_i = \sum_j W^{(2)}_{ij} z^{(1)}_j + b^{(2)}_i$.

**The collapse, worked.**
Substitute and swap the (finite) sums:

$$z^{(2)}_i
= \sum_j W^{(2)}_{ij} \Big( \sum_k W^{(1)}_{jk} x_k + b^{(1)}_j \Big) + b^{(2)}_i
= \sum_k \underbrace{\Big( \sum_j W^{(2)}_{ij} W^{(1)}_{jk} \Big)}_{W^{\mathrm{eff}}_{ik}} x_k
+ \underbrace{\sum_j W^{(2)}_{ij} b^{(1)}_j + b^{(2)}_i}_{b^{\mathrm{eff}}_i}.$$

One affine map, with effective weights $W^{\mathrm{eff}}$ and bias
$b^{\mathrm{eff}}$.
Nothing about depth changes the argument: a third layer collapses onto the
first two, and so on — **any** activation-free stack equals a single affine
layer (p11 writes the induction and its consequences out as a graded proof).

**Numeric instance.**
$W^{(1)} = \begin{pmatrix} 1 & 2 \\ 0 & -1 \end{pmatrix}$,
$b^{(1)} = (1, -2)$,
$W^{(2)} = \begin{pmatrix} 3 & -1 \end{pmatrix}$,
$b^{(2)} = (0.5)$:

$$W^{\mathrm{eff}} = (3 \cdot 1 - 1 \cdot 0,\;\; 3 \cdot 2 - 1 \cdot (-1)) = (3, 7),
\qquad b^{\mathrm{eff}} = 3 \cdot 1 + (-1)(-2) + 0.5 = 5.5 .$$

In [ ]:
W1 = np.array([[1.0, 2.0], [0.0, -1.0]])
b1 = np.array([1.0, -2.0])
W2 = np.array([[3.0, -1.0]])
b2 = np.array([0.5])

W_eff = np.array([[3.0, 7.0]])       # computed by hand above
b_eff = np.array([5.5])

rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (100, 2))

z1 = (X[:, None, :] * W1[None, :, :]).sum(axis=2) + b1     # layer 1
z2 = (z1[:, None, :] * W2[None, :, :]).sum(axis=2) + b2    # layer 2, no activation
direct = (X[:, None, :] * W_eff[None, :, :]).sum(axis=2) + b_eff

print("max |stack - single affine| :", np.abs(z2 - direct).max())

The max gap prints at float round-off scale ($\approx 3.6 \times 10^{-15}$;
the two routes merely add the same terms in different orders): on 100 random
inputs the two-layer stack and the single affine map are the same function.

**The moral, stated once and used all unit:**
depth without activations is cosmetic.
An activation-free network — however deep — has one straight decision
boundary and can represent only affine functions.
The step, tanh, and ReLU inserted *between* layers are what let networks
bend: corners, regions, staircases.
Session 2 exploits exactly this with the step activation.

### Checkpoint 5

1. Compute $W^{\mathrm{eff}}$ and $b^{\mathrm{eff}}$ by hand for
   $W^{(1)} = \begin{pmatrix} 2 & 0 \\ 1 & 1 \end{pmatrix}$,
   $b^{(1)} = (0, 1)$, $W^{(2)} = (1, 2)$, $b^{(2)} = (-1)$.
2. True or false, one reason: some 10-layer activation-free network
   represents $f(x) = |x|$.
3. Exactly where does the Section 5 substitution argument *break* if a step
   activation is applied between the layers?

## 6. Worked Exam-Style Example 1: Normal-Form Multiple Choice

Hand-running a tiny network is natural exam material, wrapped in the numeric
normal form so exactly one decoded answer is right.
Here is one in full, in the real register.

---

**Problem (reasoning is not required; no code needed).**
A network takes $x = (1, 4)$.
The hidden layer has weights
$W^{(1)} = \begin{pmatrix} 2 & -1 \\ 1 & 1 \end{pmatrix}$ (row $j$ holds the
weights of hidden unit $j$), bias $b^{(1)} = (-3, -2)$, and **step**
activation ($1$ if the pre-activation is $\ge 0$, else $0$).
The output is affine — no activation — with weights $w^{(2)} = (5, 3)$ and
bias $b^{(2)} = -\tfrac32$.
The output $y$ is a positive rational; write it in lowest terms $p/q$ with
$\gcd(p, q) = 1$, $q > 0$.
What is $p + q$?

A. 4  B. 5  C. 15  D. 17  E. 25

---

**Solution.**

*Step 1 — hidden pre-activations.*
$z^{(1)}_1 = 2 \cdot 1 - 1 \cdot 4 - 3 = -5$;
$z^{(1)}_2 = 1 \cdot 1 + 1 \cdot 4 - 2 = 3$.

*Step 2 — step activation.*
$h = (\mathrm{step}(-5), \mathrm{step}(3)) = (0, 1)$.

*Step 3 — output.*
$y = 5 \cdot 0 + 3 \cdot 1 - \tfrac32 = \tfrac32$.

*Step 4 — decode the normal form.*
$\tfrac32$ is already in lowest terms: $p + q = 3 + 2 = 5$: **answer B**.
The traps are built in: forgetting $b^{(2)}$ gives $3 = 3/1$, i.e. $4$
(choice A); feeding $x$ in transposed order $(4, 1)$ gives
$h = (1, 1)$, $y = 6.5 = 13/2$, i.e. $15$ (choice C); using ReLU instead of
the stated step gives $h = (0, 3)$, $y = 7.5 = 15/2$, i.e. $17$ (choice D);
failing to reduce $1.5 = 15/10$ gives $25$ (choice E).

*Step 5 — the free cross-check.*
Three lines of NumPy re-run the arithmetic:

In [ ]:
x = np.array([[1.0, 4.0]])
W1_mc = np.array([[2.0, -1.0], [1.0, 1.0]])
b1_mc = np.array([-3.0, -2.0])
w2_mc = np.array([[5.0, 3.0]])
b2_mc = np.array([-1.5])

h = step_activation((x[:, None, :] * W1_mc[None, :, :]).sum(axis=2) + b1_mc)
y = (h[:, None, :] * w2_mc[None, :, :]).sum(axis=2) + b2_mc
print("h =", h[0], "  y =", y[0, 0], "  -> 3/2, p + q = 5")

### Checkpoint 6

1. Redo the problem with ReLU at the hidden layer (everything else
   unchanged): what are $h$, $y$, and $p + q$?
2. Redo it with input $x = (2, 1)$ and the step activation: compute $y$.
   Does the normal form still decode cleanly, and to what?

## 7. Common Pitfalls I

**Pitfall 1 — `np.max` is not `np.maximum`.**
`np.maximum(z, 0.0)` is elementwise ReLU.
`np.max(z, 0)` is an **aggregation**: it reads the `0` as *axis 0* and
returns the largest entry — a scalar where you wanted an array, with no
error to warn you.

In [ ]:
z_bug = np.array([-2.0, 1.0, 4.0, -0.5])
print("np.maximum(z, 0.0):", np.maximum(z_bug, 0.0), "   <- ReLU, elementwise")
print("np.max(z, 0)      :", np.max(z_bug, 0), "   <- BROKEN: axis-0 aggregation, a single number")

The broken call silently returns `4.0` — downstream code then broadcasts a
scalar where a $(4,)$ array belonged, and the shape bug surfaces far from
its cause.

**Pitfall 2 — `>` where the register says `>=`.**
The two versions of the step differ on exactly one input, $z = 0$ — and
$z = 0$ happens *constantly* in designed networks, because hand-picked
integer weights put test points right on boundaries.

In [ ]:
z_edge = np.array([-1.0, 0.0, 2.0])
print("(z >= 0):", (z_edge >= 0).astype(float), "   <- the unit's pinned convention")
print("(z >  0):", (z_edge > 0).astype(float), "   <- BROKEN: silently flips every boundary point")

Identical except at the boundary — outputs $(0, 1, 1)$ versus $(0, 0, 1)$.
On a graded region-membership problem where the polygon's edges belong to
the region, the strict version misclassifies every point on an edge
(p15 dissects a full incident).

**Pitfall 3 — booleans are not the contracted floats.**
`(z >= 0)` alone returns booleans.
They *look* fine in a print, and mixed arithmetic often upcasts them — until
an operation refuses:

In [ ]:
a = np.array([1.0, -1.0]) >= 0
c = np.array([-2.0, 3.0]) >= 0
print("a:", a, " dtype:", a.dtype)
try:
    diff = a - c          # BROKEN: numpy refuses to subtract booleans
except TypeError as e:
    print("a - c raises TypeError:", e)
print("fixed:", a.astype(float) - c.astype(float),
      " dtype:", (a.astype(float)).dtype)

The subtraction raises `TypeError` — NumPy refuses `-` on boolean arrays.
That is why `step_activation` ends with `.astype(float)` and why exam
contracts state the output dtype: return the floats you promised.

### Checkpoint 7

1. A classmate's "ReLU" turns a `(50,)` pre-activation array into a single
   number. Which pitfall, and what is the one-token fix?
2. Another classmate's region code disagrees with the answer key on exactly
   the points lying on the region's edges. Which pitfall, and which
   convention does this course pin?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $(1, -2)$: $z = 2 + 2 - 3 = 1 \ge 0$ → output 1.
   $(2, 1)$: $z = 4 - 1 - 3 = 0$ → output 1 (boundary fires).
2. $w = (1)$, $b = -2$: $z = x_1 - 2 \ge 0 \iff x_1 \ge 2$.
3. Every $z$ is multiplied by 10; no output changes, because thresholding
   at 0 only reads the *sign* of $z$, which positive scaling preserves.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $(0, 1, 1, 0)$.
2. $z = 2 \cdot 3 - 1 - 3 = 2 \ge 0$ — it fires.
3. The step is constant near $z = 1.7$, so its derivative there is 0, and
   the chain rule multiplies every downstream gradient by that 0.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Step: $(0, 1, 1)$; relu: $(0, 0, 2)$; tanh:
   $(-0.762, 0, 0.964)$ at $z = -1, 0, 2$ respectively.
2. Bounded: step and tanh. Differentiable everywhere: only tanh (step
   jumps at 0; relu has a kink at 0).
3. $1 - \tanh^2(0) = 1 - 0 = 1$: near 0, tanh has slope 1 and value 0, so
   $\tanh z \approx z$ — almost the identity for small $|z|$.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $g(-2) = 0 - 0 = 0$; $g(0) = 1 - 0 = 1$; $g(4) = 5 - 3 = 2$.
   Piecewise: 0 for $z \le -1$; slope 1 from $(-1, 0)$ up to $(1, 2)$;
   constant 2 for $z \ge 1$.
2. $\mathrm{relu}(z) - \mathrm{relu}(z - 3)$; the plateau value is $3$.
3. Kinks at $z = 0, 2, 5$; slopes $0$, then $1$, then $1 + 4 = 5$, then
   $5 - 1 = 4$.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. $W^{\mathrm{eff}} = (1 \cdot 2 + 2 \cdot 1,\; 1 \cdot 0 + 2 \cdot 1)
   = (4, 2)$; $b^{\mathrm{eff}} = 1 \cdot 0 + 2 \cdot 1 - 1 = 1$.
2. False: every activation-free stack equals one affine map, and $|x|$ is
   not affine (its slope changes at 0).
3. At the substitution step: with a step in between, layer 2 reads
   $\mathrm{step}(z^{(1)})$, not $z^{(1)}$, and
   $\mathrm{step}$ of an affine expression is no longer affine, so the sums
   cannot be swapped into a single weighted sum of the $x_k$.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $h = (\mathrm{relu}(-5), \mathrm{relu}(3)) = (0, 3)$;
   $y = 9 - 1.5 = 7.5 = 15/2$; $p + q = 17$.
2. $z^{(1)} = (2 \cdot 2 - 1 - 3,\; 2 + 1 - 2) = (0, 1)$ — the first
   unit sits on the boundary and **fires**: $h = (1, 1)$;
   $y = 5 + 3 - 1.5 = 6.5 = 13/2$, decoding to $p + q = 15$.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Pitfall 1: `np.max` aggregated over axis 0. Fix: `np.maximum` (with the
   `0.0` second argument).
2. Pitfall 2: the code thresholds with `>` while the course register pins
   `>=` — boundary points ($z = 0$) belong to the firing side.

</details>